Use this notebook to:
-Download all PDB structures from SabDab
-Preprocess and Extract relevant sequence and structural information about the nanobodies/antibodies.
-Create required json files to train/evaluate the tools used in the study

In [ ]:
# use nanodesigner1 kernel
import json
import os
import sys
import concurrent.futures
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import partial
import shutil
from shutil import rmtree
nanodesigner_folder = "./NanoDesigner" # UPDATE
sys.path.append(nanodesigner_folder)
from functionalities.nanobody_antibody_interacting_residues import interacting_residues
sys.path.append(f'{nanodesigner_folder}/dyMEAN') 
from configs import IMGT, Chothia
from data.pdb_utils import Protein, AgAbComplex2, Peptide
from define_aa_contacts_antibody_nanobody import interacting_residues
from utils.renumber import renumber_pdb
from utils.network import url_get


In [ ]:
#Download and unzip data from the SabDab database:

!wget https://opig.stats.ox.ac.uk/webapps/newsabdab/sabdab/archive/all/ -O ../all_structures.zip
!unzip all_structures.zip

#Download the inforation asociated to the structure data:

!wget https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/summary/all/ -O ../tsv_summary.tsv
!mv ../tsv_summary.tsv {nanodesigner_folder}/all_structures

"""
Expected Outcome:

├── all_structures
│   ├── chothia
│   ├── imgt
│   ├── raw
    └── tsv_summary.tsv
"""

In [ ]:
# Define paths to the TSV and output folders

tsv_file_path = "./NanoDesigner/all_structures/tsv_summary.tsv" # UPDATE
output_folder_path = './NanoDesigner/built_datasets'
os.makedirs(output_folder_path, exist_ok=True)

# Define parameters
nb_or_ab = "Antibody"  # Choose "Nanobody" or "Antibody"
numbering = "imgt"  # Choose numbering scheme ("imgt" or "chothia")

# for Step 1
json_file = os.path.join(output_folder_path, f"valid_tsv_lines.json")
txt_file = os.path.join(output_folder_path, f"tsv_stats.txt")

# for Step 2
cdr_json_file = os.path.join(output_folder_path, f"{nb_or_ab}_with_cdr_info_{numbering}.json")
raw_dir = os.path.join("./NanoDesigner/all_structures", numbering)    #UPDATE

# for Step 3
filtered_json_file = os.path.join(output_folder_path, f"{nb_or_ab}_filtered_tsv_lines_{numbering}.json") # change name

# for Step 4
binding_json_file = os.path.join(output_folder_path, f"{nb_or_ab}_with_binding_info_{numbering}.json")
tmp_dir_for_analysis = os.path.join(output_folder_path, f"tmp_dir_SASA_computations")

# for Step 5
cdrh3_json_file = os.path.join(output_folder_path, f"CDRH3_interacting_{nb_or_ab}_{numbering}.json")


In [ ]:
# Helper functions

# Define a function to extract relevant information from a TSV line (for cases where the json file chain ID info may be wrong)
def extract_info_from_line(line):
    fields = line.strip().split('\t')
    pdb_id = fields[0]
    # Check if the line has enough fields
    if len(fields) < 2:
        # Handle the case when the line doesn't have enough fields
        return None
    # leave as they are as other methods may fail due to chain ID inconsistencies 'i' instead of "I" and so on
    heavy_chain = fields[1]#.upper # some entries ID's may be in lower case and then wont be found
    light_chain = fields[2]
    antigen_chains = [chain for chain in fields[4].split(' | ')]
    antigen_type = [type.strip() for type in fields[5].split('|')]
    resolution = fields[16]  # Add this line to extract the resolution field

    # Check if any field is "NA" and replace it with an empty string
    if pdb_id == "NA":
        pdb_id = ""
    if heavy_chain == "NA":
        heavy_chain = ""
    if light_chain == "NA":
        light_chain = ""
    if "NA" in antigen_chains:
        antigen_chains = ["" if chain == "NA" else chain for chain in antigen_chains]
    if "NA" in antigen_type:
        antigen_type = ["" if type == "NA" else type for type in antigen_type]

    #if pdb_id or heavy_chain are empty or NaN, dont give a dicitonary
    if not pdb_id or pdb_id == "NA" or not heavy_chain or heavy_chain == "NA":
        return

    return {
        'pdb': pdb_id,
        'heavy_chain': heavy_chain,
        'light_chain': light_chain,
        'antigen_chains': antigen_chains,
        'antigen_type': antigen_type,
        'resolution': resolution,  # Add this line to include the resolution field in the return dictionary
    }

def fetch_from_sabdab(identifier, numbering_scheme, save_path, tries=5):
    # example identifier: 3ogo
    # example numbering_scheme: 'imgt' or 'chothia'

    print(f"fetching {identifier} from sabdab")

    identifier = identifier.lower()
    url = f'https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabdab/pdb/{identifier}/?scheme={numbering_scheme}'

    text = url_get(url, tries)
    if text is None:
        return None

    with open(save_path, 'w') as file:
        file.write(text.text)

    data = {
        'pdb': save_path
    }
    return data

def filter_entries(entry, flag):
    pdb = entry.get('pdb')
    heavy_chain = entry.get('heavy_chain')
    light_chain = entry.get('light_chain')
    antigen_chains = entry.get('antigen_chains')
    antigen_type = entry.get('antigen_type')
    resolution = entry.get('resolution')
    # resolution = float(entry['resolution'])

    try:
        resolution = float(resolution)
    except (ValueError, KeyError):
        # resolution = np.nan  # Assign NaN if resolution is missing or not a number
        resolution = 0.0

    if flag == "Nanobody":
        # Keep if heavy chain is present, light chain is empty, resolution <= 4.0, and antigen type is protein or peptide
        if heavy_chain != '' and light_chain == '' and resolution <= 4.0 and ('protein' in antigen_type or 'peptide' in antigen_type):
            return entry
        else:
            return None

    if flag == "Antibody":
        # Keep if both heavy and light chains are present, resolution <= 4.0, and antigen type is protein or peptide
        if heavy_chain != '' and light_chain != '' and resolution <= 4.0 and ('protein' in antigen_type or 'peptide' in antigen_type):
            return entry
        else:
            return None

 
def binding_residues_analysis(inputs,item):
    # item = json.loads(item)

    tmp_dir_for_interacting_aa, json_file4  = inputs
    pdb_file = item["pdb_data_path"]
    heavy_chain = H =item["heavy_chain"]
    antigen_chains = item["antigen_chains"]

    cdrh3_seq = item["cdrh3_seq"]
    cdrh2_seq = item["cdrh2_seq"]
    cdrh1_seq = item["cdrh1_seq"] 

    # Get the base name of the file
    filename = os.path.basename(pdb_file)

    # Split the filename into name and extension
    model_name, extension = os.path.splitext(filename)

    epitope_model = []
    cdr1_interactions_to_ag = []
    cdr2_interactions_to_ag = []
    cdr3_interactions_to_ag = []

    print(f"Starting time before binding interface analysis")
    for antigen in antigen_chains:
        print(f"Processing {antigen}")
        # the program doesnt work with PDBs with one more than one chain in the PDB; create a temp pdb with the current pdb
        if len(antigen_chains) > 1:
            tmp_pdb = os.path.join(tmp_dir_for_interacting_aa, f"{model_name}_chain{antigen}.pdb")
            chains_to_reconstruct = []
            chains_to_reconstruct.extend(H)
            chains_to_reconstruct.extend(antigen)
            # program crashes with more than 2 chains, as we only care about cdrh involvement we will skip reconstruction of light chain in this temporal file
            # if L:
            #     chains_to_reconstruct.extend(L)

            if not os.path.exists(tmp_pdb):
                try:
                    protein = Protein.from_pdb(pdb_file, chains_to_reconstruct)
                    protein.to_pdb(tmp_pdb)

                    # Use assert to check if tmp_pdb exists
                    assert os.path.exists(tmp_pdb), f"Temporary PDB file not created: {tmp_pdb}"

                except Exception as e:
                    print(f"First attempt failed to process PDB file '{pdb_file}': {e}")


            result = interacting_residues(item, tmp_pdb, antigen, tmp_dir_for_interacting_aa)
            if result is not None:
                # epitope_result, cdr3_matching_res, cdr2_matching_res, cdr1_matching_res = interacting_residues(item, tmp_pdb, antigen,tmp_dir)
                epitope_result, cdr3_matching_res, cdr2_matching_res, cdr1_matching_res = result
            else:
                continue
        else:
            result = interacting_residues(item, pdb_file, antigen, tmp_dir_for_interacting_aa)
            if result is not None:
                # epitope_result, cdr3_matching_res, cdr2_matching_res, cdr1_matching_res = interacting_residues(item, tmp_pdb, antigen,tmp_dir)
                epitope_result, cdr3_matching_res, cdr2_matching_res, cdr1_matching_res = result
            else:
                continue

        epitope_model.extend(epitope_result) 
        cdr3_interactions_to_ag.extend(cdr3_matching_res)
        cdr2_interactions_to_ag.extend(cdr2_matching_res)
        cdr1_interactions_to_ag.extend(cdr1_matching_res)

    cdr1_interactions_to_ag = [list(t) for t in set(tuple(i) for i in cdr1_interactions_to_ag)]
    cdr2_interactions_to_ag = [list(t) for t in set(tuple(i) for i in cdr2_interactions_to_ag)]
    cdr3_interactions_to_ag = [list(t) for t in set(tuple(i) for i in cdr3_interactions_to_ag)]


    cdr3_involvement = (len(cdr3_interactions_to_ag)/len(cdrh3_seq))*100 if len(cdrh3_seq) != 0.0 else 0.0
    cdr2_involvement = (len(cdr2_interactions_to_ag)/len(cdrh2_seq))*100 if len(cdrh2_seq) != 0.0 else 0.0
    cdr1_involvement = (len(cdr1_interactions_to_ag)/len(cdrh1_seq))*100 if len(cdrh1_seq) != 0.0 else 0.0
    total_avg_cdr_involvement = float((cdr3_involvement + cdr2_involvement + cdr1_involvement) / 3)

    epitope_model = [list(t) for t in set(tuple(i) for i in epitope_model)]

    item["epitope"] = epitope_model
    item["cdr1_interactions_to_ag"] = cdr1_interactions_to_ag
    item["cdr2_interactions_to_ag"] = cdr2_interactions_to_ag
    item["cdr3_interactions_to_ag"] = cdr3_interactions_to_ag
    item["total_avg_cdr_involvement"] = total_avg_cdr_involvement
    item["cdr3_avg"] = cdr3_involvement
    item["cdr2_avg"] = cdr2_involvement
    item["cdr1_avg"] = cdr1_involvement

    with open(json_file4,'a') as f:
        f.write(json.dumps(item) + '\n')

In [ ]:
# Step 1: Extract information from the TSV file
# This step reads each line from the input TSV file, validates it, and converts each valid entry into a JSON format.
# Valid entries are saved to an intermediate JSON file for further processing.


if not os.path.exists(json_file) or os.stat(json_file).st_size == 0:
    all_tsv_lines = 0
    valid_lines = 0
    invalid_lines = 0
    with open(tsv_file_path, 'r') as f:
        next(f)
        for line in f:
            all_tsv_lines += 1
            info_dict = extract_info_from_line(line)
            if info_dict is not None:
                valid_lines += 1
                with open(json_file, 'a') as json_out:
                    json_out.write(json.dumps(info_dict) + '\n')
            else:
                invalid_lines += 1

    with open(txt_file, 'w') as stats_file:
        stats_file.write(f"Total lines: {all_tsv_lines}\nValid lines: {valid_lines}\nInvalid lines: {invalid_lines}\n")

print(f"Step 1 completed: Valid lines saved to {json_file} file.")


In [ ]:
# Step 2: Filter complete entries
# This step filters the entries from Step 1 based on specific criteria for either Nanobody or Antibody.
# Only entries with acceptable resolution, chain configuration, and antigen type are retained.
# Filtered entries are saved to a separate JSON file.

if not os.path.exists(filtered_json_file) or os.stat(filtered_json_file).st_size == 0:
    with open(json_file, 'r') as f:
        tsv_lines = f.read().strip().split('\n')
    
    filtered_entries = []
    for line in tsv_lines:
        entry = json.loads(line)
        filtered_entry = filter_entries(entry, nb_or_ab)
        if filtered_entry:
            filtered_entries.append(filtered_entry)
            with open(filtered_json_file, 'a') as out_f:
                out_f.write(json.dumps(filtered_entry) + '\n')

print(f"Step 2 completed: Filtered entries saved to {filtered_json_file}")


In [ ]:
# Step 3: CDR Information Extraction and PDB Reconstruction
# This step extracts Complementarity-Determining Region (CDR) information and reconstructs PDB files for specific antibody/nanobody entries.
# For each entry, it checks if the CDR information file is empty or missing, then reads each entry from a filtered JSON file.
# It verifies the presence of the raw PDB file or downloads it from SabDab if necessary.
# Each entry is enriched with a unique path to the reconstructed PDB file containing only specified chains.
# The final output includes enriched CDR information and paths to reconstructed PDB files, saved in a designated output directory.

output_folder_specific = os.path.join(output_folder_path, f"{nb_or_ab}_{numbering}")
os.makedirs(output_folder_specific, exist_ok=True)


# Initialize the output file if it doesn't exist
if not os.path.exists(cdr_json_file) or os.stat(cdr_json_file).st_size == 0:
    with open(cdr_json_file, 'w') as out_f:
        pass  # Create the file if it doesn't exist

# Load filtered entries
with open(filtered_json_file, 'r') as f:
    filtered_entries = f.read().strip().split('\n')

pdb_counter = defaultdict(int)  # Dictionary to keep track of counts per pdb_id

def process_entry(line):
    """Function to process each entry in parallel."""
    try:
        # Parse the JSON line to get the item dictionary
        item = json.loads(line)
        pdb_id = item['pdb']
        current_raw_path = os.path.join(raw_dir, f"{pdb_id}.pdb")

        H = item["heavy_chain"]
        L = item["light_chain"] if item["light_chain"] else "X"  # Default to "X" if light_chain is empty
        Ag = item["antigen_chains"]
        ordered_ag = ''.join(sorted(Ag))  # Sort antigen chains alphabetically
        item["entry_id"] = f"{pdb_id}_{H}_{L}_{ordered_ag}"  # Create entry_id based on formatted strings

        # Increment counter for the specific pdb_id
        pdb_counter[pdb_id] += 1
        pdb_suffix = pdb_counter[pdb_id]  # Unique suffix for this pdb_id instance

        # Fetch the PDB file from SabDab if it does not exist locally
        if not os.path.exists(current_raw_path):
            fetch_from_sabdab(pdb_id, numbering, current_raw_path)

        # Proceed only if the PDB file is available
        if os.path.exists(current_raw_path):
            # Create the AgAbComplex2 object for CDR extraction
            cplx = AgAbComplex2.from_pdb(
                current_raw_path, 
                item['heavy_chain'], 
                item['light_chain'], 
                item['antigen_chains'], 
                numbering=numbering, 
                skip_epitope_cal=True
            )

            # Extract sequences for heavy chain, light chain, and antigens
            item['heavy_chain_seq'] = cplx.get_heavy_chain().get_seq() if cplx.get_heavy_chain() else ""
            item['light_chain_seq'] = cplx.get_light_chain().get_seq() if cplx.get_light_chain() else ""
            item['antigen_seqs'] = [chain.get_seq() for _, chain in cplx.get_antigen() if chain]

            # Extract CDR positions and sequences for heavy and light chains
            for c in ['H', 'L']:
                for i in range(1, 4):
                    cdr_name = f'{c}{i}'.lower()
                    cdr = cplx.get_cdr(cdr_name)
                    if cdr:
                        item[f'cdr{cdr_name}_pos'] = cplx.get_cdr_pos(cdr_name)
                        item[f'cdr{cdr_name}_seq'] = cdr.get_seq()
                    else:
                        item[f'cdr{cdr_name}_pos'] = []
                        item[f'cdr{cdr_name}_seq'] = ""

            # Add numbering scheme to the item
            item["numbering"] = numbering

            # Construct the pdb_data_path with the unique suffix and add it to the item
            pdb_data_filename = f"{pdb_id}_{pdb_suffix}.pdb"
            item["pdb_data_path"] = os.path.join(output_folder_specific, pdb_data_filename)

            # Save the reconstructed PDB to output_folder_specific
            if not os.path.exists(item["pdb_data_path"]):
                protein = Protein.from_pdb(current_raw_path, [item['heavy_chain']] + item['antigen_chains'] + ([item['light_chain']] if item['light_chain'] else []))
                protein.to_pdb(item["pdb_data_path"])

            # Return the item to be written to the output JSON file
            return item

    except Exception as e:
        print(f"Error processing {pdb_id}: {e}")
        return None

# Use ThreadPoolExecutor for parallel processing
with ThreadPoolExecutor() as executor:
    # Submit tasks to executor
    futures = [executor.submit(process_entry, line) for line in filtered_entries]
    
    # Write results as they complete
    with open(cdr_json_file, 'a') as out_f:
        for future in as_completed(futures):
            result = future.result()
            if result:
                out_f.write(json.dumps(result) + '\n')

print(f"Step 3 completed: CDR information saved to {cdr_json_file}")


In [ ]:
# Step 4: Binding Residues Analysis
# This step analyzes the interactions between the CDR regions and antigen residues to identify binding residues.
# For each entry, it loads the CDR data, antigen chains, and associated PDB file.
# It then calculates the involvement of CDR residues in binding to the antigen and records the positions and types of interactions.
# The results include epitope data, interaction counts for each CDR, and average CDR involvement percentages.
# All processed entries are saved to a final JSON file, with each entry enriched by binding site information.


# Ensure the temporary directory exists
if not os.path.exists(tmp_dir_for_analysis):
    os.makedirs(tmp_dir_for_analysis)

# Load entries from cdr_json_file
with open(cdr_json_file, 'r') as f:
    entries_cdr_info = [json.loads(line) for line in f.read().strip().split('\n')]

# Load processed entries from binding_json_file if it exists
processed_pdbs = set()
if os.path.exists(binding_json_file) and os.stat(binding_json_file).st_size > 0:
    with open(binding_json_file, 'r') as f:
        for line in f:
            entry = json.loads(line)
            processed_pdbs.add(entry["entry_id"])  # this is unique ID per PDB

# Filter entries to process: exclude those already in binding_json_file
entries_to_process = [entry for entry in entries_cdr_info if entry["entry_id"] not in processed_pdbs]

if entries_to_process:
    inputs = (tmp_dir_for_analysis, binding_json_file)
    process_with_args = partial(binding_residues_analysis, inputs)
    
    with concurrent.futures.ProcessPoolExecutor() as executor:
        futures = [executor.submit(process_with_args, entry) for entry in entries_to_process]
        for future in concurrent.futures.as_completed(futures):
            try:
                future.result()  # Wait for each future to complete
            except Exception as e:
                print(f"An error occurred during binding residue analysis: {e}")

    print(f"Processed {len(entries_to_process)} new entries and saved to {binding_json_file}")
else:
    print("No new entries to process. All entries in cdr_json_file are already in binding_json_file.")


In [ ]:
# Step 5: Print stats and keep only entries interacting from the CDRH3.

# Initialize counters and sets
unique_pdb_parts = set()
num_entries_with_binding = 0
num_entries_with_cdrh3_contact = 0
bound_entries = []

# Parse the antibody file to gather statistics and collect entries for CDRH3-specific JSON
with open(binding_json_file, 'r') as f:
    data = [json.loads(line) for line in f]

for entry in data:
    pdb = entry["pdb"]
    pdb_parts = pdb.rsplit('_', 1)[0]
    unique_pdb_parts.add(pdb_parts)

    # Count entries with binding and CDRH3 interaction
    if float(entry["total_avg_cdr_involvement"]) != 0.0:
        num_entries_with_binding += 1
    if float(entry["cdr3_avg"]) != 0.0:
        num_entries_with_cdrh3_contact += 1
        bound_entries.append(entry)

# Calculate the number of unique PDB IDs
num_unique_pdbs = len(unique_pdb_parts)

# Print statistics
print(f"The number of unique PDB parts in the data is: {num_unique_pdbs}")
print(f"Total number of entries: {len(data)}")
print(f"The number of entries with binding is: {num_entries_with_binding}")
print(f"The number of entries with contact in the CDRH3 region is: {num_entries_with_cdrh3_contact}")

# Save entries interacting from the CDRH3 to a new JSON file in a specified format
with open(cdrh3_json_file, 'w') as j:
    for entry in bound_entries:
        # pdb = entry["pdb"]
        # pdbcode = pdb.rsplit('_', 1)[0]
        # entry["pdb"] = pdbcode
        # H = entry["heavy_chain"]
        # L = entry["light_chain"] or "X"
        # Ag = entry["antigen_chains"]
        # ordered_ag = ''.join(sorted(Ag))
        # entry["entry_id"] = f"{pdbcode}_{H}_{L}_{ordered_ag}"
        json_data = json.dumps(entry)
        j.write(json_data + "\n")

all_tsv_lines = 0
valid_lines = 0
invalid_lines = 0

with open(tsv_file_path, 'r') as f:
    next(f)  # Skip header
    for line in f:
        all_tsv_lines += 1
        # Assuming `extract_info_from_line` is defined to parse TSV lines as required
        info_dict = extract_info_from_line(line)
        if info_dict is not None:
            valid_lines += 1
            with open(json_file, 'a') as json_out:
                json_out.write(json.dumps(info_dict) + '\n')
        else:
            invalid_lines += 1

#Save TSV statistics to a text file
stats_file_path = f"{output_folder_path}/{nb_or_ab}_dataset_statistics.txt"  
with open(stats_file_path, 'w') as stats_file:
    stats_file.write(f"Total lines from tsv file (Nb+Ab): {all_tsv_lines}\n")
    stats_file.write(f"Valid lines (Nb+Ab): {valid_lines}\n")
    stats_file.write(f"Invalid lines (Nb+Ab): {invalid_lines}\n")
    stats_file.write(f"\nThe number of unique {nb_or_ab} PDB parts in the data is: {num_unique_pdbs}\n")
    stats_file.write(f"Total number of {nb_or_ab} entries: {len(data)}\n")
    stats_file.write(f"The number of {nb_or_ab} entries with binding is: {num_entries_with_binding}\n")
    stats_file.write(f"The number of {nb_or_ab} entries with contact in the CDRH3 region is: {num_entries_with_cdrh3_contact}\n")



In [ ]:
# Step 6 (For DiffAb): Merge folders and updte file paths prior data split and training.
#Run the analysis for antibodies and nanobodies using the chothia numbering
#For easiness, and for when training DiffAb on Nanobodies + Antibodies dataset 1) merge the two folders in a unique one; 2) update the fullpath location in the json files to use

new_dir = os.path.join(output_folder_path, "all_chothia_pdbs")
nano_chothia_dir = "folder_name_for_nano_chothia"
antibody_chothia_dir = "folder_name_for_antibody_chothia"

#path to json files:
nano_chothia_json = "./built_datasets/CDRH3_interacting_Nanobody_chothia.json" #update accordingly
antibody_chothia_json = "./built_datasets/CDRH3_interacting_Antibody_chothia.json" #update accordingly
combined_json_path = os.path.join(new_dir, "CDRH3_interacting_Nano_Ab_chothia.json")

# Create a new folder with all pdbs: merge nano_chothia_dir and antibody_chothia_dir
os.makedirs(new_dir, exist_ok=True)

def copy_pdb_files(source_dir, target_dir):
    for file_name in os.listdir(source_dir):
        if file_name.endswith(".pdb"):  # Assuming PDB files have .pdb extension
            source_file = os.path.join(source_dir, file_name)
            target_file = os.path.join(target_dir, file_name)
            shutil.copy2(source_file, target_file)

copy_pdb_files(nano_chothia_dir, new_dir)
copy_pdb_files(antibody_chothia_dir, new_dir)

# Creaate a json file wiht all entries and updated "pdb_data_path" key/value; keep name but update the rest: new_dir + file name (including file extension)
def update_json_paths(json_file, target_dir):
    with open(json_file, 'r') as f:
        # data = json.load(f)
        data = [json.loads(line) for line in f]
    
    for entry in data:
        pdb_name = os.path.basename(entry["pdb_data_path"])
        entry["pdb_data_path"] = os.path.join(target_dir, pdb_name)
    
    return data

nano_data = update_json_paths(nano_chothia_json, new_dir)
antibody_data = update_json_paths(antibody_chothia_json, new_dir)

combined_data = nano_data + antibody_data

# Save combined JSON
with open(combined_json_path, 'w') as f:
    json.dump(combined_data, f)
    for item in combined_data:
        f.write(json.dumps(item) + '\n')
